# Lecture 1 (optional): fitting models to colloid data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/blob/main/L1_fitting/L1_fitting_colloids.ipynb)

<a rel="license" href="https://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://licensebuttons.net/l/by-sa/4.0/88x31.png" title='This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.' align="right"/></a>

Author: Dr Antonia Mey
Email: antonia.mey@ed.ac.uk

© Antonia Mey, University of Edinburgh, 2026. Text CC BY-SA 4.0; code cells MIT (see `LICENSE` and `LICENSE-CODE` in the repository).

**Time needed:** about 30–45 minutes. This notebook is optional practice for Lecture 1.

### Learning objectives
By the end of this notebook, you should be able to
* Fit a non-linear model to data with `scipy.optimize.curve_fit`, choose a sensible starting guess, and read the parameter uncertainties.
* Explain why fitting a linearised model (e.g. the logarithm of the data) can give a different answer.
* Use a residual plot to check a fit.
* Chain two fits together (decay rate → diffusion coefficient → particle size) and fit a straight line with `scikit-learn`.

### Table of Contents

0. [Colloids, Boltzmann and Stokes–Einstein](#0-background)
1. [Perrin's experiment: counting Avogadro's number](#1-perrin)
2. [Dynamic light scattering: one decay curve](#2-dls-one)
3. [From decay rates to particle size](#3-dls-size)
4. [Key points](#4-key-points)

### The data

Both examples come from the colloids part of Advanced Physical Chemistry (Prof. Camp's notes, sections 4.3 and 5).

* **Section 1:** Jean Perrin's measurements of gamboge grains settling in water (1909) which won him the 1926 Nobel Prize in Physics.
* **Sections 2–3:** dynamic light scattering (DLS) on silica particles in water. The data is simulated, with realistic noise, so that we know the right answer. The monodisperse sample is the silica sol from the example in section 5.4 of the colloids notes.

### How to open this notebook

* **Noteable (recommended):** open it from the Lecture 1 page on Learn. Everything you need is already installed.
* **Google Colab (backup):** click the *Open in Colab* button at the top. You need a Google account. Save a copy to your Drive (*File → Save a copy in Drive*) or your answers will be lost.

**<span style="color:black">Jupyter Cheat Sheet</span>**
- To run the currently highlighted cell and move focus to the next cell, hold <kbd>&#x21E7; Shift</kbd> and press <kbd>&#x23ce; Enter</kbd>;
- To get help for a specific function, place the cursor within the function's brackets, hold <kbd>&#x21E7; Shift</kbd>, and press <kbd>&#x21E5; Tab</kbd>;
- Cells marked `# FIXME` are for you to complete.

# Import libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression

BLUE, ORANGE, LAVENDER = "#2a78d6", "#eb6834", "#a897d1"

# physical constants (SI units)
kB = 1.380649e-23        # Boltzmann constant / J K^-1
R = 8.314462618          # gas constant / J K^-1 mol^-1
g = 9.80665              # acceleration due to gravity / m s^-2

# The DLS data files live in the data/ folder of the course repository.
# If they are not found locally (e.g. on Colab), they are downloaded from GitHub.
DATA_URL = "https://raw.githubusercontent.com/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/main/data/"


def load(name):
    try:
        return pd.read_csv("../data/" + name)
    except FileNotFoundError:
        return pd.read_csv(DATA_URL + name)

# 0. Colloids, Boltzmann and Stokes–Einstein <a id="0-background"></a>

**Sedimentation equilibrium** (notes, eq. 23). Colloidal particles in a column of liquid follow a Boltzmann distribution in height $h$:
$$\frac{n(h)}{n(0)} = \exp\left(-\frac{h}{l_g}\right), \qquad l_g = \frac{k_BT}{v_p(\rho_p - \rho_s)\,g},$$
where $l_g$ is the *gravitational length*, $v_p = \frac{4}{3}\pi r^3$ the particle volume and $\rho_p - \rho_s$ the density difference between particle and solvent.

**Stokes–Einstein** (notes, eq. 31). A sphere of radius $r$ in a solvent of viscosity $\eta$ diffuses with
$$D = \frac{k_BT}{6\pi\eta r}.$$

In both cases the quantity we want ($k_B$, or the radius $r$) is hidden inside a parameter of a curve that we fit to measurements. 

# 1. Perrin's experiment finds Avogadro's number <a id="1-perrin"></a>

In 1909 Jean Perrin prepared grains of gamboge (a yellow tree resin) of radius $r = 0.212\ \mu$m, let them settle in water at 20 °C, and counted 13,000 grains under a microscope in a cell 100 µm deep. His counts at four heights, scaled so that the lowest level is 100:

| height $h$ / µm | 5 | 35 | 65 | 95 |
|---|---|---|---|---|
| relative number of grains | 100 | 47 | 22.6 | 12 |

The density difference between gamboge and water was $\rho_p - \rho_s = 0.2067$ g cm⁻³.

If we fit $n(h) = n_0\, e^{-h/l_g}$ to these four points, we get $l_g$, hence $k_BT = l_g v_p (\rho_p-\rho_s) g$, hence $k_B$, and finally **Avogadro's constant** $N_A = R/k_B$, from counting grains in a microscope.

*Source: J. Perrin, "Brownian movement and molecular reality" (1909).*

In [ ]:
h = np.array([5, 35, 65, 95]) * 1e-6      # m
n = np.array([100, 47, 22.6, 12])          # relative counts

r_gamboge = 0.212e-6                        # m
drho = 0.2067e3                             # kg m^-3  (0.2067 g cm^-3)
T_perrin = 293.15                           # K


def boltzmann(h, n0, lg):
    """Sedimentation equilibrium: number of grains at height h."""
    return n0 * np.exp(-h / lg)

`curve_fit(model, x, y, p0=...)` finds the parameters that minimise the sum of squared residuals (the MSE loss from the lecture). Because the model is *not linear* in $l_g$, there is no formula for the answer: `curve_fit` searches iteratively, starting from the guess `p0`. It returns the best parameters `popt` and their covariance matrix `pcov`; the square roots of the diagonal of `pcov` are the standard uncertainties of the parameters.

## Tasks 1 <a id="tasks-1"></a>
1. Fit `boltzmann` to Perrin's data with `curve_fit`. Use a starting guess read off the table (the counts roughly halve every 30 µm). Print $l_g$ with its uncertainty in µm, and plot the data with the fitted curve.
2. Calculate $k_B$ and $N_A$ from your $l_g$. How close are you to today's value, $N_A = 6.022 \times 10^{23}$ mol⁻¹? Perrin himself reported $7.05 \times 10^{23}$.
3. Before computers, the standard way was to linearise: $\ln n = \ln n_0 - h/l_g$ is a straight line in $h$. Fit a straight line to $\ln n$ against $h$ (e.g. with `np.polyfit(h, np.log(n), 1)`) and get $l_g$ and $N_A$ again. Why is the answer not the same as in task 1?
4. The uncertainty in $l_g$ from task 1 is about 2 %, yet $N_A$ is about 20 % too high. So the error is not random scatter in the four points. $N_A$ depends on the grain radius as $1/r^3$. By how much would Perrin's radius have to be wrong to explain the whole discrepancy?

In [ ]:
# 1. Non-linear fit of Perrin's data
# FIXME

In [ ]:
# 2. Boltzmann and Avogadro constants
# FIXME

In [ ]:
# 3. Linearised fit
# FIXME

In [ ]:
# 4. How wrong would the radius have to be?
# FIXME

*Your answers to questions 3 and 4 here.*

# 2. Dynamic light scattering: one decay curve <a id="2-dls-one"></a>

In DLS (colloids notes, section 4.3) a laser shines through the sample and a detector records how the scattered intensity flickers as the particles diffuse. The instrument computes a correlation function $g(\tau)$. Small particles move fast, so the correlation dies away quickly. For identical spheres
$$g(\tau) = \beta\, e^{-\Gamma \tau}, \qquad \Gamma = D q^2,$$
where $\beta \le 1$ is an instrument constant and $q$ the size of the scattering vector at scattering angle $\theta$,
$$q = \frac{4\pi n_s}{\lambda}\sin\frac{\theta}{2}.$$
Here $\lambda = 633$ nm is the laser wavelength in vacuum and $n_s = 1.332$ the refractive index of water.

**Sample:** silica particles in water at 298 K, measured at 9 angles from 30° to 150°. We start with the measurement at 90°.

In [ ]:
mono = load("dls_silica_monodisperse.csv")
tau = mono["tau_s"].values           # s
g90 = mono["g_90deg"].values

T, eta = 298.15, 8.90e-4             # K, Pa s (water)
n_s, lam = 1.332, 633e-9


def q_of(theta_deg):
    """Scattering vector / m^-1."""
    return 4 * np.pi * n_s / lam * np.sin(np.radians(theta_deg) / 2)


def decay(tau, beta, Gamma):
    """Single-exponential correlation function."""
    return beta * np.exp(-Gamma * tau)


fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(tau, g90, ".", color=BLUE)
ax.set_xlabel("lag time τ / s"); ax.set_ylabel("g(τ)"); ax.set_title("silica, 90°")
plt.show()

The lag times are spaced evenly on a log scale, from 1 µs to 30 ms, as in a real instrument.
## Tasks 2 <a id="tasks-2"></a>
1. Read a starting guess off the plot: $g$ falls to about $1/e$ of its starting value at $\tau \approx 1/\Gamma$. Fit `decay` with `curve_fit` and print $\beta$ and $\Gamma$ with uncertainties.
2. Plot the residuals $g - g_\text{fit}$ against $\tau$ (log axis). For a good model they should scatter randomly around zero, with no pattern. Is that the case? What is the MSE, compared with the noise level of the instrument (standard deviation 0.005, so variance $2.5\times 10^{-5}$)?

In [ ]:
# 1. Non-linear fit
# FIXME

In [ ]:
# 2. Residuals
# FIXME

### What a bad starting guess does

The cell below repeats the fit with `p0=[1, 1e8]`, a guess that is out by a factor of about $10^4$ (easily done by mixing up units). Run it and look at the "fit".

In [ ]:
# A bad starting guess
with warnings.catch_warnings():
    warnings.simplefilter("ignore")          # curve_fit warns that it cannot estimate the covariance
    p_bad, _ = curve_fit(decay, tau, g90, p0=[1, 1e8])
print("result:", p_bad)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(tau, g90, ".", color=BLUE, label="data")
ax.semilogx(tau, decay(tau, *p_bad), color=ORANGE, label="'fit'")
ax.set_xlabel("τ / s"); ax.set_ylabel("g(τ)"); ax.legend()
plt.show()

`curve_fit` returns the starting guess unchanged, without an error. With $\Gamma = 10^8$ s⁻¹ the model has decayed to zero long before the first lag time (1 µs), so it predicts $g = 0$ for every data point. Changing $\Gamma$ a little still gives zero everywhere: the loss is flat, its gradient is zero, and gradient-based search has no direction to go in. This is the non-convex loss landscape from the lecture: the starting point matters. Always plot the fit, and choose `p0` from the data.

# 3. From decay rates to particle size <a id="3-dls-size"></a>

One angle gives one $\Gamma$. Measuring at several angles checks the model. If the particles really diffuse, then $\Gamma = D q^2$, so $\Gamma$ plotted against $q^2$ is a *straight line through the origin* with slope $D$. Then Stokes–Einstein gives the radius, $r = k_BT / 6\pi\eta D$.

## Tasks 3 <a id="tasks-3"></a>
1. Loop over the 9 angles (the columns `g_30deg`, …, `g_150deg`): fit `decay` to each and collect the 9 values of $\Gamma$ in an array. (Your starting guess has to work for all angles. Use the value from Section 2 for every angle, or scale it with $q^2$.)
2. Plot $\Gamma$ against $q^2$. Is it a straight line through the origin?
3. Fit the straight line with `scikit-learn`: `LinearRegression(fit_intercept=False)`, which forces the line through the origin. Note that `scikit-learn` wants the input as a 2D array (`q2.reshape(-1, 1)`). Read off $D$ from `model.coef_[0]`.
4. Calculate the particle radius and diameter in nm. Compare $D$ and $d$ with the silica example in section 5.4 of the colloids notes.

In [ ]:
# 1. Fit every angle
angles = np.arange(30, 151, 15)
# FIXME

In [ ]:
# 2. Gamma against q^2
# FIXME

In [ ]:
# 3. Straight line through the origin with scikit-learn
# FIXME

In [ ]:
# 4. Particle size
# FIXME

# 4. Key points <a id="4-key-points"></a>

* Fitting = minimising a loss. `curve_fit` minimises the sum of squared residuals by an iterative search from a starting guess `p0`.
* **Choose `p0` from the data** and always **plot the fit**: a bad guess can leave the search stuck where the loss is flat, without an error message.
* **Do not linearise: it changes the loss.** Fitting $\ln y$ gives the small values much more weight, so the answer changes.
* The **parameter uncertainties** from `pcov` describe the scatter of the data about the model. They do not include errors in the constants you put in.
* **Residual plots** check a fit: they should scatter randomly around zero. A pattern means the model is missing something.
* Fits can be chained: decay curves → $\Gamma$ at each angle → $D$ from a straight line ($\Gamma = Dq^2$) → radius from Stokes–Einstein.